In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression
from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
from catboost import CatBoostRegressor
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
#回归问题评价指标
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error 
from sklearn.model_selection import cross_val_score
#分类问题评价指标
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss#对数损失（交叉熵损失）
random_seed = 123
np.random.seed(random_seed)
#读取数据
data=pd.read_excel('DATA.xlsx',header=None)
y=data.iloc[:,-1].values
# X=pd.read_csv('pca_X.csv',index_col=0).values
X=data.iloc[:,:-1].values
# 将数据分为训练集和测试集
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=random_seed)
print('X_train.shape',X_train.shape)
print('X_test.shape',X_test.shape)
print('y_train.shape',y_train.shape)
print('y_test.shape',y_test.shape)
#位置1为训练集r2，位置2为训练集rmse，位置3为训练集mae，位置4为测试集r2，位置5为测试集rmse，位置6为测试集mae，位置7为5折交叉验证平均r2，位置8为5折交叉验证平均rmse，位置9为5折交叉验证平均mae
RandomForest_metric=[X,X.X,X,X,X,X,X,X,X] 
Svr_metric=[X,X.X,X,X,X,X,X,X,X] 
Adaboost_metric=[X,X.X,X,X,X,X,X,X,X] 
Xgboost_metric=[X,X.X,X,X,X,X,X,X,X] 
DecisionTree_metric=[X,X.X,X,X,X,X,X,X,X] 
GBDT_metric=[X,X.X,X,X,X,X,X,X,X] 
ANN_metric=[X,X.X,X,X,X,X,X,X,X] 
catboost_metric=[X,X.X,X,X,X,X,X,X,X] 
import numpy as np

# 提供的指标数据
metrics = np.array([
    [X,X.X,X,X,X,X,X,X,X] ,  # RandomForest
    [X,X.X,X,X,X,X,X,X,X] ,  # SVR
    [X,X.X,X,X,X,X,X,X,X] ,  # Adaboost
    [X,X.X,X,X,X,X,X,X,X] ,  # Xgboost
    [X,X.X,X,X,X,X,X,X,X] ,  # DecisionTree
    [X,X.X,X,X,X,X,X,X,X] ,  # GBDT
    [X,X.X,X,X,X,X,X,X,X] ,  # ANN
    [X,X.X,X,X,X,X,X,X,X]    # Catboost
])

# 负向指标的索引位置
negative_indices = [1,2, 4, 5,7, 8]  # 训练集RMSE, 训练集MAE, 测试集RMSE, 测试集MAE, 交叉验证RMSE, 交叉验证MAE

# 步骤 1: 数据标准化（最大最小值标准化）
def min_max_normalize(data, negative_indices):
    # 获取最小值和最大值
    min_vals = data.min(axis=0)
    max_vals = data.max(axis=0)
    normalized_data = (data - min_vals) / (max_vals - min_vals)
    
    # 反转负向指标
    normalized_data[:, negative_indices] = 1 - normalized_data[:, negative_indices]
    
    return normalized_data

normalized_metrics = min_max_normalize(metrics, negative_indices)

# 步骤 2: 计算熵值
def calculate_entropy(data):
    m, n = data.shape
    entropy = np.zeros(n)
    
    for j in range(n):
        # 计算每个指标的占比
        p = data[:, j] / np.sum(data[:, j])
        p = np.clip(p, 1e-10, 1)  # 避免对0取对数
        entropy[j] = -np.sum(p * np.log(p)) / np.log(m)
    
    return entropy

entropy = calculate_entropy(normalized_metrics)

# 步骤 3: 计算权重
def calculate_weights(entropy):
    # 权重与熵值的关系
    return (1 - entropy) / np.sum(1 - entropy)

weights = calculate_weights(entropy)

# 步骤 4: 计算综合得分
def calculate_scores(normalized_metrics, weights):
    return np.dot(normalized_metrics, weights)

scores = calculate_scores(normalized_metrics, weights)

# 步骤 5: 排名
rankings = np.argsort(scores)[::-1]  # 降序排列

# 输出各个基学习器的综合得分和排名
base_learners = [
    "RandomForest", "SVR", "Adaboost", "Xgboost", "DecisionTree", "GBDT", "ANN", "Catboost"
]

for i, idx in enumerate(rankings):
    print(f"{base_learners[idx]}: 综合得分 = {scores[idx]:.4f}, 排名 = {i+1}")
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# 模型名称和对应的得分
models = ['Catboost', 'GBDT', 'Xgboost', 'Adaboost', 'RandomForest', 'SVR', 'DecisionTree', 'ANN']
scores = [0.9942, 0.9797, 0.9459, 0.8956, 0.7730, 0.3876, 0.3575, 0.0031]

# 创建一个 DataFrame 来存储模型和得分
df = pd.DataFrame({
    'Model': models,
    'Score': scores
})

# 设置字体为 Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'

# 设置 Seaborn 样式
sns.set(style="whitegrid")

# 创建颜色映射：得分越高颜色越深
norm = Normalize(vmin=min(scores), vmax=max(scores))
cmap = plt.get_cmap('Blues')  # 选择蓝色渐变调色板
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# 创建条形图
plt.figure(figsize=(10, 6))
barplot = sns.barplot(x='Score', y='Model', data=df)

# 设置条形的颜色映射
for i, bar in enumerate(barplot.patches):
    bar.set_facecolor(sm.to_rgba(scores[i]))  # 根据得分设置颜色

# 在每个条形后添加得分
for i, bar in enumerate(barplot.patches):
    # 在条形右侧添加得分
    barplot.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{scores[i]:.4f}', va='center', fontsize=16, family='Times New Roman')

# 设置标题和标签，并指定字体
barplot.set_title('Entropy Weight Composite Scores', fontsize=28, fontweight='bold', family='Times New Roman')
barplot.set_xlabel('Score', fontsize=24, family='Times New Roman')
barplot.set_ylabel('Model', fontsize=24, family='Times New Roman')

# 设置模型名称的字体和大小
barplot.set_yticklabels(models, fontsize=18, family='Times New Roman')

# 设置横坐标刻度范围在 0 到 1 之间
barplot.set_xlim(0, 1)

# 修改横坐标刻度字体和大小
barplot.tick_params(axis='x', labelsize=18)
barplot.xaxis.set_tick_params(labelsize=18, labelcolor='black', labelrotation=0, direction='in', length=6)
# 显示图形
plt.show()
from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 定义基学习器（你可以根据需要调整超参数）
base_regressors = [
    ('CatBoost', CatBoostRegressor(depth=3, iterations=200, l2_leaf_reg=1, learning_rate=0.1, verbose=False)),
    ('GBDT', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)),
    ('XGBoost', XGBRegressor(colsample_bytree=0.9, gamma=0, learning_rate=0.1, max_depth=3, n_estimators=150, subsample=0.8)), 
]

# 初始化堆叠回归器
stacking_regressor = StackingRegressor(
    estimators=base_regressors,
    final_estimator=LinearRegression())
# 训练模型
stacking_regressor.fit(X_train, y_train)

# 预测训练集和测试集
y_pred_train = stacking_regressor.predict(X_train)
y_pred_test = stacking_regressor.predict(X_test)


# 计算评估指标
r2_score_train = stacking_regressor.score(X_train, y_train)
mse_train = mean_squared_error(y_train, y_pred_train)
mae_train = mean_absolute_error(y_train, y_pred_train)
r2_score_test = stacking_regressor.score(X_test, y_test)
mse_test = mean_squared_error(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test) 
# 输出结果
print("训练集R2", r2_score_train)
print("训练集均方根误差：", mse_train)
print("测试集均方根误差：", mse_test)
print("测试集R2", r2_score_test)
print("训练集MAE：",mae_train)
print("测试集MAE：",mae_test)
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 给定的评估指标
r2_value = X
rmse_value = X
mae_value = X

# 设置字体为Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'

# 创建图形
plt.figure(figsize=(5, 5), dpi=600)

# 假设你已经有了 y_train, y_pred_train, y_test, y_pred_test
# 1. 增强型散点图绘制
scatter_kws = {'s': 120, 'alpha': 0.7, 'edgecolors': 'white', 'linewidths': 0.5}
sns.scatterplot(x=y_train, y=y_pred_train, color='#B63737', marker='o', 
                label='Train Set', **scatter_kws)
sns.scatterplot(x=y_test, y=y_pred_test, color='#3D7A94', marker='s', 
                label='Test Set', **scatter_kws)

# 2. 参考线增强
max_val = max(max(y_train), max(y_test))
plt.plot([0, max_val], [0, max_val], 'k-', linewidth=1.5, alpha=0.7)  # 去掉label参数

# 3. 在参考线附近添加文字标注 y=x
plt.text(max_val * 0.3, max_val * 0.5, 'y = x', fontsize=18, color='black', ha='center', va='center')

# 4. 坐标轴设置
plt.axis('equal')
plt.xlim(0, max_val * 1.05)
plt.ylim(0, max_val * 1.05)
# 设置刻度字号
ax = plt.gca()
ax.tick_params(axis='both', which='major', labelsize=12)
# 5. 专业标注系统
metrics_text = (
    f"Test Set Metrics:\n"
    f"RMSE = {rmse_value:.2f}\n"
    f"MAE = {mae_value:.2f}\n"
    f"R² = {r2_value:.2f}\n"
)
plt.text(0.95, 0.025, metrics_text, transform=plt.gca().transAxes,
         ha='right', va='bottom', fontsize=12,
         bbox=dict(boxstyle="round,pad=0.4", facecolor='white', alpha=0.9))  # 移除边框颜色

# 修改边框颜色和宽度
ax = plt.gca()  # 获取当前坐标轴
ax.spines['top'].set_color('#444444')  # 设置上边框颜色
ax.spines['top'].set_linewidth(1.5)  # 设置上边框宽度为1

ax.spines['right'].set_color('#444444')  # 设置右边框颜色
ax.spines['right'].set_linewidth(1.5)  # 设置右边框宽度为1

ax.spines['left'].set_color('#444444')  # 设置左边框颜色
ax.spines['left'].set_linewidth(1.5)  # 设置左边框宽度为1

ax.spines['bottom'].set_color('#444444')  # 设置下边框颜色
ax.spines['bottom'].set_linewidth(1.5) # 设置下边框宽度为1

# 设置最外面的边框颜色为#444444
plt.gca().spines['top'].set_color('#444444')
plt.gca().spines['right'].set_color('#444444')
plt.gca().spines['bottom'].set_color('#444444')
plt.gca().spines['left'].set_color('#444444')

# 6. 图例与标题
plt.xlabel('True Values', fontsize=16, labelpad=10)
plt.ylabel('Predicted Values', fontsize=16, labelpad=10)
plt.legend(frameon=True, edgecolor='gray', loc='upper left', bbox_to_anchor=(0, 1))  # 图例放左上角
plt.text(0.5, 1.05, 'Model: Stacking', ha='center', va='bottom', fontsize=16, color='black', 
         transform=plt.gca().transAxes, fontweight='bold', fontname='Times New Roman')
plt.xlabel('True Values', fontsize=16, labelpad=10)
plt.ylabel('Predicted Values', fontsize=16, labelpad=10)
plt.legend(frameon=True, edgecolor='gray', loc='upper left', 
           bbox_to_anchor=(0, 1), fontsize=14)  # 字体大小设为14
# 关闭网格背景
plt.grid(False)

plt.tight_layout()
plt.savefig("Stacking.tif", format='tif', bbox_inches='tight',dpi=1200)
plt.show()
